## 실습 1: 원시 예약 데이터 수집 (Bronze 계층)

**Bronze 계층**은 원본 시스템에서 도착한 그대로의 원시 데이터를 수정 없이 저장합니다. 운영 파이프라인에서는 데이터가 자산 관리 시스템(PMS, Property Management System)에서 도착합니다. 이 실습에서는 다섯 개의 GlobStay 자산에 걸친 예약을 나타내는 샘플 데이터셋이 제공됩니다.

데이터 문제는 의도적으로 포함된 것입니다 — 이는 Silver 계층에서 해결하게 될 실제 상황을 반영합니다.

### ✅ 제공됨: Unity Catalog 구조와 샘플 데이터 생성

아래 두 셀을 실행하여 다음을 수행하세요:
- `bronze`, `silver`, `gold` 스키마를 포함하는 카탈로그 `hospitality_lab` 생성
- 의도적인 품질 문제가 포함된 샘플 원시 예약 DataFrame 생성

출력을 주의 깊게 살펴보세요 — 실습 2를 진행하기 전에 데이터 문제를 이해해야 합니다.

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_ws*")]
dplab10_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dplab10_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"스토리지 루트: {storage_root}")

In [ ]:
# ✅ 이 셀을 실행하세요 — Unity Catalog 구조를 생성합니다
# 카탈로그 이름을 변경합니다. 예시: hospitality_lab** -> hospitality_lab_alias

catalog = "hospitality_lab"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog} MANAGED LOCATION '{storage_root}' COMMENT 'GlobStay 호스피탈리티 분석 플랫폼'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze COMMENT '수집된 원시 예약 데이터'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver COMMENT '정제 및 검증된 예약 데이터'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold   COMMENT '분석 준비가 완료된 집계 데이터'")

print(f"✅ 카탈로그 '{catalog}' 및 스키마 bronze / silver / gold 가 생성되었습니다.")

In [ ]:
# ✅ 이 셀을 실행하세요 — 원시 예약 DataFrame을 생성합니다

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("booking_id",      StringType(),  True),
    StructField("property_id",     StringType(),  True),
    StructField("property_name",   StringType(),  True),
    StructField("check_in_date",   StringType(),  True),
    StructField("check_out_date",  StringType(),  True),
    StructField("room_type",       StringType(),  True),
    StructField("rate_per_night",  DoubleType(),  True),
    StructField("guest_id",        StringType(),  True),
    StructField("guest_name",      StringType(),  True),
    StructField("guest_email",     StringType(),  True),
    StructField("booking_status",  StringType(),  True),
    StructField("channel",         StringType(),  True),
    StructField("adults",          IntegerType(), True),
    StructField("children",        IntegerType(), True),
])

raw_data = [
    # (booking_id, property_id, property_name, check_in, check_out, room_type, rate, guest_id, guest_name, guest_email, status, channel, adults, children)
    ("B001","P01","Grand Harbor Hotel",   "2025-06-15","2025-06-18","Deluxe",  250.00,"G001","Alex Rivera",      "alex.r@email.com",    "confirmed","direct",    2,0),
    ("B002","P01","Grand Harbor Hotel",   "2025-06-20","2025-06-22","Suite",   450.00,"G002","Maria Santos",     "m.santos@corp.com",   "confirmed","corporate", 1,0),
    ("B003","P02","Sunset Beach Resort",  "2025-07-01","2025-07-08","Sea View",180.00,"G003","James Chen",       "jchen@email.com",     "confirmed","ota",       2,2),
    ("B004","P02","Sunset Beach Resort",  "2025-07-10","2025-07-12","Garden",  140.00,"G004","Sarah Johnson",    None,                  "confirmed","direct",    2,0),  # 이메일 null
    ("B005","P03","Mountain Pines Lodge", "2025-08-05","2025-08-10",None,       120.00,"G005","Robert Kim",       "rkim@email.com",      "cancelled","ota",       1,0),  # room_type null
    ("B006","P03","Mountain Pines Lodge", "2025-08-15","2025-08-12","Standard",110.00,"G006","Emily Brown",      "ebrown@email.com",    "confirmed","direct",    2,1),  # check_out < check_in
    ("B001","P01","Grand Harbor Hotel",   "2025-06-15","2025-06-18","Deluxe",  250.00,"G001","Alex Rivera",      "alex.r@email.com",    "confirmed","direct",    2,0),  # B001 중복
    ("B007","P04","City Center Inn",      "2025-09-10","2025-09-11","Standard",-50.00,"G007","Michael Turner",   "mturner@email.com",   "confirmed","corporate", 1,0),  # 음수 rate
    ("B008","P04","City Center Inn",      "2025-09-15","2025-09-18","Superior",195.00,"G008","Lisa Wong",        "lwong@email.com",     "confirmed","ota",       2,0),
    ("B009","P01","Grand Harbor Hotel",   "2025-10-01","2025-10-05","Suite",   480.00,"G009","Carlos Mendez",    "cmendes@biz.com",     "confirmed","corporate", 2,0),
    ("B010","P02","Sunset Beach Resort",  "2025-11-20","2025-11-25","Sea View",155.00,"G010","Aisha Patel",      "aisha.p@email.com",   "confirmed","direct",    2,1),
    ("B011","P03","Mountain Pines Lodge", "2025-12-22","2025-12-26","Deluxe",  180.00,"G011","Thomas Muller",    "tmuller@email.com",   "confirmed","ota",       2,0),
    ("B012","P04","City Center Inn",      "2025-10-08","2025-10-09","Standard",  0.00,"G012","Yuki Tanaka",      "ytanaka@email.com",   "confirmed","direct",    1,0),  # 0 rate
    ("B013","P01","Grand Harbor Hotel",   "2025-07-04","2025-07-07","Deluxe",  275.00,"G013","Sophie Anderson",  "sanderson@travel.com","confirmed","ota",       2,2),
    ("B014","P02","Sunset Beach Resort",  "2025-08-20","2025-08-27","Suite",   350.00,"G014","Daniel Hart",      "dhart@email.com",     "pending",  "direct",    2,0),
    ("B015","P05","Airport Transit Hotel","2025-06-30","2025-07-01","Standard", 99.00,"G015","Priya Sharma",     None,                  "confirmed","ota",       1,0),  # 이메일 null
]

raw_df = spark.createDataFrame(raw_data, schema)
raw_df.display()
print(f"\n원시 레코드 수: {raw_df.count()}")

### 🎯 챌린지: 원시 데이터를 Bronze 테이블에 작성하기

`raw_df`를 `hospitality_lab.bronze` 스키마의 **`bookings_bronze`**라는 이름의 Delta 테이블에 작성하세요.

**요구 사항:**
- 셀을 다시 실행할 수 있도록 `overwrite` 모드를 사용하세요
- 작성 후, 레코드 수가 원시 DataFrame과 일치하는지 확인하세요

> 🤖 **Genie Code 팁:** *"saveAsTable과 overwrite 모드를 사용하여 PySpark DataFrame을 Unity Catalog 테이블에 작성하려면 어떻게 하나요?"*라고 물어보세요.

In [ ]:
# 여기에 코드를 작성하세요
# raw_df를 hospitality_lab.bronze.bookings_bronze에 작성하세요 (overwrite 모드)
# 그런 다음 레코드 수를 확인하세요


## 실습 2: 예약 데이터 정제 및 검증 (Silver 계층)

**Silver 계층**은 다운스트림 프로세스가 신뢰할 수 있는 정제·검증된 데이터를 포함합니다. bronze 계층에 존재하는 각 품질 문제를 반드시 해결해야 합니다:

| 문제 | 발견 위치 | 규칙 |
|---|---|---|
| 중복 예약 | B001이 두 번 나타남 | `booking_id`별로 첫 번째 항목만 유지 |
| Null `room_type` | B005 | room type이 없는 레코드 제외 |
| 잘못된 날짜 순서 | B006 (`check_out`이 `check_in`보다 앞섬) | `check_out_date <= check_in_date`인 레코드 제외 |
| 0 이하의 요금 | B007 (–50), B012 (0) | `rate_per_night <= 0`인 레코드 제외 |

> **참고:** Null `guest_email` 값(B004, B015)은 허용됩니다 — 모든 예약에 이메일 주소가 필요한 것은 아닙니다.

### 🎯 챌린지: Silver 테이블 생성하기

`hospitality_lab.bronze.bookings_bronze`에서 읽어와 네 가지 정제 규칙을 모두 적용한 뒤, 그 결과를 **`hospitality_lab.silver.bookings_silver`**에 작성하세요 (overwrite 모드).

**힌트:**
- 중복을 제거하려면 `.dropDuplicates(["booking_id"])`를 사용하세요
- null을 제거하려면 `.filter(col("room_type").isNotNull())`를 사용하세요
- 날짜를 비교하려면 `pyspark.sql.functions`에서 `to_date`와 `datediff`를 임포트하세요
- 각 규칙에 대해 `.filter()` 호출을 체이닝하세요

작성 후, 레코드가 제거되었는지 확인하기 위해 Bronze와 Silver의 레코드 수를 모두 출력하세요.

> 🤖 **Genie Code 팁:** *"날짜 비교를 사용하여 check_out_date가 check_in_date보다 이후가 아닌 PySpark 행을 필터링하려면 어떻게 하나요?"*라고 물어보세요.

In [ ]:
# 여기에 코드를 작성하세요
# bronze에서 읽어와 정제 규칙을 적용한 뒤 silver에 작성하세요
# bronze 레코드 수와 silver 레코드 수를 출력하세요


## 실습 3: 분석 준비 데이터 구축 (Gold 계층)

**Gold 계층**은 고도로 정제된, 비즈니스용 데이터셋을 포함합니다. GlobStay 경영진은 두 개의 분석 테이블을 원합니다:

1. **`property_performance`** — 호텔 자산별 총 확정 예약 수, 판매된 숙박일 수, 매출
2. **`channel_performance`** — 예약 채널(direct, OTA, corporate)별로 구분된 예약 건수와 매출

### 🎯 챌린지 3a: 자산 성과 테이블

Spark SQL을 사용하여 Gold 테이블 `hospitality_lab.gold.property_performance`를 생성하세요.

**요구 사항:**
- 소스: `hospitality_lab.silver.bookings_silver`
- `booking_status = 'confirmed'`인 레코드만 필터링
- 컬럼: `property_id`, `property_name`, `confirmed_bookings`, `total_nights`, `total_revenue`
- `DATEDIFF(check_out_date, check_in_date)`를 사용하여 `total_nights` 계산
- `total_revenue`를 `total_nights × rate_per_night`로 계산
- `total_revenue DESC`로 정렬

> 🤖 **Genie Code 팁:** *"DATEDIFF와 SUM을 사용하여 자산별 예약 매출을 집계하는 Spark SQL CREATE OR REPLACE TABLE 문을 작성해줘."*라고 물어보세요.

In [ ]:
# 여기에 코드를 작성하세요 — %sql 또는 spark.sql() 사용
# hospitality_lab.gold.property_performance를 생성하세요


### 🎯 챌린지 3b: 예약 채널 성과 테이블

각 예약 채널이 GlobStay 매출에 어떻게 기여하는지 보여주는 Gold 테이블 `hospitality_lab.gold.channel_performance`를 생성하세요.

**요구 사항:**
- 소스: `hospitality_lab.silver.bookings_silver`
- `booking_status = 'confirmed'`인 레코드만 필터링
- 컬럼: `channel`, `booking_count`, `total_revenue`, `avg_revenue_per_booking` (소수점 둘째 자리로 반올림)
- `total_revenue DESC`로 정렬

> 🤖 **Genie Code 팁:** *"ROUND와 AVG를 사용하여 Spark SQL에서 예약당 평균 매출을 계산하려면 어떻게 하나요?"*라고 물어보세요.

In [ ]:
# 여기에 코드를 작성하세요
# hospitality_lab.gold.channel_performance를 생성하세요


## 실습 4: 파이프라인 노트북의 오류 처리

노트북이 Lakeflow Job 태스크로 실행될 때는 오류를 우아하게 처리하고 그 결과를 다운스트림 태스크에 전달해야 합니다. 두 가지 핵심 패턴은 다음과 같습니다:

1. **`PySparkException`을 활용한 `try/except`** — 특정 Spark 오류를 포착하여 노트북을 중단시키지 않고 처리
2. **`dbutils.notebook.exit()`** — Job이 검사할 수 있는 구조화된 메시지로 성공 또는 실패를 신호

### 🎯 챌린지 4a: try/except로 안전한 테이블 읽기 구현

다음을 수행하는 함수 `safe_read_table(table_name)`을 작성하세요:
1. `spark.table(table_name)`을 사용하여 테이블 읽기를 시도
2. `pyspark.errors`의 `PySparkException`을 포착
3. 테이블을 찾을 수 없는 경우 오류 클래스를 포함한 설명 메시지를 출력
4. 오류 시 `None`을, 성공 시 DataFrame을 반환

유효한 테이블과 존재하지 않는 테이블로 테스트하여 두 경로가 모두 동작하는지 확인하세요.

**힌트:**
- 임포트: `from pyspark.errors import PySparkException`
- 구조화된 오류 코드를 얻으려면 `e.getErrorClass()`를 사용하세요
- 처리하지 않은 예기치 않은 오류는 다시 발생(re-raise)시키세요

> 🤖 **Genie Code 팁:** *"Python에서 PySparkException을 포착하고 그 오류 클래스를 출력하는 방법을 보여줘."*라고 물어보세요.

In [ ]:
# 여기에 코드를 작성하세요
# try/except를 사용하여 safe_read_table(table_name)을 정의하세요
# 유효한 테이블 이름과 존재하지 않는 테이블 이름으로 테스트하세요


### 🎯 챌린지 4b: dbutils.notebook.exit()로 태스크 결과 신호하기

노트북이 Lakeflow Job 태스크로 실행될 때, `dbutils.notebook.exit()`를 호출하면 다운스트림 태스크가 `{{tasks.<task_name>.values.<key>}}`를 사용하여 결과를 읽을 수 있습니다.

다음을 수행하는 함수 `report_pipeline_outcome(table_name)`을 작성하세요:
1. 지정된 테이블을 읽고 레코드 수를 셈
2. 성공 시, `"SUCCESS: Processed <n> records from <table_name>"` 메시지와 함께 `dbutils.notebook.exit()`를 호출
3. 어떤 예외든 발생 시, `"FAILED: <error message>"` 메시지와 함께 `dbutils.notebook.exit()`를 호출

> ⚠️ **경고:** `dbutils.notebook.exit()`는 노트북 실행을 즉시 중단시킵니다. 함수는 정의하되 **호출하지 마세요** — 대신 호출 부분을 주석 처리된 상태로 두세요. Job에서는 이를 호출하는 것이 올바른 동작입니다.

> 🤖 **Genie Code 팁:** *"노트북이 Job 태스크로 실행될 때 Azure Databricks에서 dbutils.notebook.exit는 어떻게 동작하나요?"*라고 물어보세요.

In [ ]:
# 여기에 코드를 작성하세요
# report_pipeline_outcome(table_name)을 정의하세요
# 함수 호출은 주석 처리된 상태로 두세요


## 실습 5: 매개변수화된 노트북 태스크

재사용 가능한 파이프라인 노트북은 런타임에 매개변수를 받습니다 — 코드 변경 없이 동일한 노트북이 다양한 시나리오(예: 서로 다른 날짜, 환경, 필터 값)에서 동작하도록 만듭니다.

Lakeflow Jobs에서는 두 가지 메커니즘이 사용됩니다:
- **`dbutils.widgets`** — Job 태스크 구성에서 노트북 *안으로* 매개변수를 전달
- **`dbutils.jobs.taskValues`** — 동일한 Job 실행 내에서 노트북 태스크 *사이로* 값을 전달

### 🎯 챌린지 5a: 런타임 위젯 매개변수 추가

기본값 `confirmed`와 레이블 `Booking Status Filter`를 가진 `booking_status`라는 이름의 텍스트 위젯을 추가하세요.

그런 다음 위젯 값을 가져와 `hospitality_lab.silver.bookings_silver`를 필터링하는 데 사용하세요. 필터링된 결과를 표시하고 레코드 수를 출력하세요.

**힌트:**
- `dbutils.widgets.text(name, default, label)`
- `dbutils.widgets.get(name)`은 현재 값을 반환합니다
- Lakeflow Job에서는 태스크 **Parameters** 구성이 런타임에 기본값을 재정의합니다

> 🤖 **Genie Code 팁:** *"Databricks에서 텍스트 위젯을 생성하고 그 값을 가져와 DataFrame을 필터링하려면 어떻게 하나요?"*라고 물어보세요.

In [ ]:
# 여기에 코드를 작성하세요
# 위젯 'booking_status'를 추가하세요 (기본값: 'confirmed', 레이블: 'Booking Status Filter')
# 값을 가져와 bookings_silver를 필터링한 뒤 결과를 표시하세요


### 🎯 챌린지 5b: 다운스트림 태스크로 값 전달하기

처리가 끝난 후, 노트북 태스크는 `dbutils.jobs.taskValues.set()`을 사용하여 계산된 값을 다운스트림 태스크와 공유할 수 있습니다.

Silver 테이블에서 두 개의 태스크 값을 설정하세요:
1. 키 `silver_record_count` — `bookings_silver`의 레코드 수
2. 키 `properties_processed` — 고유한 `property_id` 값들의 Python 리스트

그런 다음 설정한 내용을 출력하여 확인할 수 있도록 하세요.

> **참고:** `dbutils.jobs.taskValues`는 Lakeflow Job 실행 내부에서만 사용할 수 있습니다. 대화형 노트북 세션에서는 경고를 출력하고 아무 작업도 하지 않습니다(no-op) — 이는 예상된 동작입니다.

> 🤖 **Genie Code 팁:** *"dbutils.jobs.taskValues.set을 사용하여 한 Databricks 노트북 태스크에서 다른 태스크로 리스트 값을 전달하려면 어떻게 하나요?"*라고 물어보세요.

In [ ]:
# 여기에 코드를 작성하세요
# 태스크 값을 설정하세요: 'silver_record_count'와 'properties_processed'
# 그런 다음 값을 출력하여 확인하세요
